# Efficient Operations, Broadcasting, and Conditional Transformations

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'customer_id': range(501, 513),
    'income': [25000, 42000, 18000, 65000, 50000, 0,
               38000, 72000, 29000, 55000, 46000, 33000],
    'expenses': [15000, 28000, 12000, 40000, 30000, 5000,
                 22000, 45000, 18000, 35000, 26000, 20000],
    'orders': [2, 5, 1, 8, 6, 0, 3, 9, 2, 7, 4, 3],
    'discount_rate': [0.05, 0.10, 0.00, 0.15, 0.10, 0.00,
                      0.05, 0.20, 0.00, 0.15, 0.10, 0.05]
})
df

,customer_id,income,expenses,orders,discount_rate
0,501,25000,15000,2,0.05
1,502,42000,28000,5,0.10
2,503,18000,12000,1,0.00
3,504,65000,40000,8,0.15
4,505,50000,30000,6,0.10
5,506,0,5000,0,0.00
6,507,38000,22000,3,0.05
7,508,72000,45000,9,0.20
8,509,29000,18000,2,0.00
9,510,55000,35000,7,0.15


#### Q1. Vectorized arithmetic

Create savings by subtracting expenses from income. Then create annual_savings by multiplying monthly savings by 12. Do this without loops.

In [5]:
df['savings']=df['income']-df['expenses']

#### Q2. Broadcasting with a scalar

Assume every customer receives a monthly bonus of ₹2,000. Create adjusted_income by adding this bonus to each customer's income. Explain how broadcasting works in this operation.

In [12]:
df['adjusted_income']=df['income']+2000


#### Q3. Conditional transformation with np.where()

Create income_status as follows:
- Income greater than or equal to 50,000: High
- Otherwise: Standard

In [17]:
df['income_status']=np.where(df['income']>=50000,'High','Standard')

#### Q4. Multiple conditions with np.select()

Create financial_segment using these rules:
- Income below 25,000: Low
- Income from 25,000 to below 45,000: Medium
- Income from 45,000 to below 65,000: High
- Income of 65,000 or above: Very High

In [20]:
conditions=[
    df['income']<25000,
    df['income'].between(25000,44999),
    df['income'].between(45000,64999),
    df['income']>=65000
]
labels=['Low','Medium','High','Very High']
df['financial_segment']=np.select(conditions,labels,default='n/a')


#### Q5. Vectorized discount calculation

Create discount_amount by multiplying income by discount_rate. Then create income_after_discount by subtracting the discount amount from income.

In [25]:
df['discount_amount']=df['income']-df['discount_rate']
df['income_after_discount']=df['income']-df['discount_amount']

#### Q6. Safe division

Create expense_ratio as expenses divided by income, expressed as a percentage. Use vectorized operations and ensure zero income does not produce infinity.

In [30]:
df['expense_ratio']=(df['expenses']/df['income']).where(df['income']!=0,np.nan)*100

#### Q7. Conditional clipping

Create capped_expenses by limiting expenses to a maximum of ₹30,000 using a vectorized Pandas or NumPy operation. Keep the original expenses column unchanged.

In [35]:
df['capped_expenses']=df['expenses'].clip(upper=30000)
df

,customer_id,income,expenses,orders,discount_rate,savings,adjusted_income,income_status,financial_segment,discount_amount,income_after_discount,expense_ratio,capped_expenses
0,501,25000,15000,2,0.05,10000,27000,Standard,Medium,24999.95,0.05,60.000000,15000
1,502,42000,28000,5,0.10,14000,44000,Standard,Medium,41999.90,0.10,66.666667,28000
2,503,18000,12000,1,0.00,6000,20000,Standard,Low,18000.00,0.00,66.666667,12000
3,504,65000,40000,8,0.15,25000,67000,High,Very High,64999.85,0.15,61.538462,30000
4,505,50000,30000,6,0.10,20000,52000,High,High,49999.90,0.10,60.000000,30000
5,506,0,5000,0,0.00,-5000,2000,Standard,Low,0.00,0.00,NaN,5000
6,507,38000,22000,3,0.05,16000,40000,Standard,Medium,37999.95,0.05,57.894737,22000
7,508,72000,45000,9,0.20,27000,74000,High,Very High,71999.80,0.20,62.500000,30000
8,509,29000,18000,2,0.00,11000,31000,Standard,Medium,29000.00,0.00,62.068966,18000
9,510,55000,35000,7,0.15,20000,57000,High,High,54999.85,0.15,63.636364,30000


#### Q8. Compare vectorized operations and apply()

Create savings_category with these rules:
- Savings below 10,000: Low Savings
- Savings from 10,000 to below 25,000: Moderate Savings
- Savings of 25,000 or above: High Savings
Solve it using vectorized conditions rather than row-wise apply(). Explain one advantage of vectorization.

In [38]:
conditions = [
    df['savings'] < 10000,
    (df['savings'] >= 10000) & (df['savings'] < 25000),
    df['savings'] >= 25000
]

choices = [
    'Low Savings',
    'Moderate Savings',
    'High Savings'
]

df['savings_category'] = np.select(
    conditions,
    choices,
    default='Unknown'
)

#### Q9. Conditional replacement

Create valid_income by replacing incomes less than or equal to zero with NaN. Then create income_after_tax, assuming a 10% tax rate for valid incomes. Use vectorized operations.

In [41]:
df['valid_income']=np.where(
                    df['income']<=0,np.nan,df['income'])

df['income_after_tax']=df['valid_income']*0.9
df

,customer_id,income,expenses,orders,discount_rate,savings,adjusted_income,income_status,financial_segment,discount_amount,income_after_discount,expense_ratio,capped_expenses,savings_category,valid_income,income_after_tax
0,501,25000,15000,2,0.05,10000,27000,Standard,Medium,24999.95,0.05,60.000000,15000,Moderate Savings,25000.0,22500.0
1,502,42000,28000,5,0.10,14000,44000,Standard,Medium,41999.90,0.10,66.666667,28000,Moderate Savings,42000.0,37800.0
2,503,18000,12000,1,0.00,6000,20000,Standard,Low,18000.00,0.00,66.666667,12000,Low Savings,18000.0,16200.0
3,504,65000,40000,8,0.15,25000,67000,High,Very High,64999.85,0.15,61.538462,30000,High Savings,65000.0,58500.0
4,505,50000,30000,6,0.10,20000,52000,High,High,49999.90,0.10,60.000000,30000,Moderate Savings,50000.0,45000.0
5,506,0,5000,0,0.00,-5000,2000,Standard,Low,0.00,0.00,NaN,5000,Low Savings,NaN,NaN
6,507,38000,22000,3,0.05,16000,40000,Standard,Medium,37999.95,0.05,57.894737,22000,Moderate Savings,38000.0,34200.0
7,508,72000,45000,9,0.20,27000,74000,High,Very High,71999.80,0.20,62.500000,30000,High Savings,72000.0,64800.0
8,509,29000,18000,2,0.00,11000,31000,Standard,Medium,29000.00,0.00,62.068966,18000,Moderate Savings,29000.0,26100.0
9,510,55000,35000,7,0.15,20000,57000,High,High,54999.85,0.15,63.636364,30000,Moderate Savings,55000.0,49500.0
